# Phase 3 - Skin-Condition Classifier Training
## Google SCIN + Fitzpatrick17k via HuggingFace (streaming)

**Key fix:** `streaming=True` means NO 13 GB download.
Images are processed one at a time directly from HuggingFace parquet shards.
Free Colab (15 GB disk) works fine.

**One-time setup before running:**
1. Accept SCIN terms at https://huggingface.co/datasets/google/scin
2. Accept Fitzpatrick17k (search `mattgroh/fitzpatrick17k` on HF)
3. Get a **read token** from https://huggingface.co/settings/tokens
4. Push repo to GitHub, fill `GITHUB_USER` in Cell 1

| Source | Total images | After face/neck filter |
|---|---|---|
| `google/scin` | ~10k | ~1-3k |
| `fitzpatrick17k` | ~16k | ~2-4k |
| **Combined** | **~26k scanned, ~13 GB streamed** | **~3-7k saved** |

## 0. Confirm GPU

In [14]:
!nvidia-smi | head -10
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

Mon Apr 27 09:05:11 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
CUDA: True Tesla T4


## 1. Clone repo + install

In [15]:
GITHUB_USER = "moodykhalif23"
BRANCH = "main"

!git clone -b {BRANCH} https://github.com/moodykhalif23/facemeup.git /content/skincare
%cd /content/skincare/backend_v2

!pip install -q -e ml_service
!pip install -q -e ml_training   # includes datasets, huggingface-hub
print("install done")

fatal: destination path '/content/skincare' already exists and is not an empty directory.
/content/skincare/backend_v2
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for skincare-ml-service (pyproject.toml) ... done
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for skincare-ml-training (pyproject.toml) ... done
install done


## 2. Set HuggingFace token

> **Why not `login()`?**  
> When running Colab from the VS Code extension, the vault secret injection
> is unavailable. Set the token directly via `os.environ` instead.

Paste your **read token** from https://huggingface.co/settings/tokens
(it starts with `hf_`)

In [ ]:
import os
from getpass import getpass

# Retrieve HF token from environment or prompt user
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    HF_TOKEN = getpass('Paste your HuggingFace token (hf_...): ')

os.environ['HF_TOKEN'] = HF_TOKEN

# Verify it works
from huggingface_hub import whoami
try:
    info = whoami(token=HF_TOKEN)
    print('Logged in as:', info['name'])
except Exception as e:
    print('Auth error:', e)
    print('Check your token at https://huggingface.co/settings/tokens')

Logged in as: rikamaze023


## 3. Inspect SCIN schema + face-region coverage

SCIN uses **multi-hot boolean columns** for body parts:
  `body_parts_head_or_neck`, `body_parts_arm`, `body_parts_palm`, etc.
This cell shows which columns exist and how many rows are face/neck.

In [17]:
from datasets import load_dataset

ds = load_dataset("google/scin", split="train", streaming=True, token=HF_TOKEN)

cols = set(ds.features.keys())
print('--- All columns ---')
for col in sorted(cols):
    print(f"  {col}")

# SCIN multi-hot body-part columns
body_cols = sorted(c for c in cols if c.startswith('body_parts_'))
FACE_BP   = {'body_parts_head_or_neck','body_parts_face','body_parts_scalp','body_parts_neck'}
face_cols = [c for c in body_cols if c in FACE_BP]
print(f'\nFace body-part columns found: {face_cols}')
print(f'All body-part columns: {body_cols}')

# Sample first 500 rows to show face-region rate
face_n = total_n = 0
for i, row in enumerate(ds):
    if i >= 500: break
    total_n += 1
    if face_cols and any(str(row.get(c,'')).lower() in ('true','1') for c in face_cols):
        face_n += 1
    elif not face_cols:
        face_n += 1   # no body-part info — count all
print(f"\nFirst 500 rows: {face_n} face/neck ({100*face_n/max(1,total_n):.0f}%)")
print("Extrapolating to full 10k dataset: ~", round(face_n/500*10000), "face images")

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Some datasets params were ignored: ['splits', 'download_size', 'dataset_size']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

--- All columns ---
  age_group
  body_parts_arm
  body_parts_back_of_hand
  body_parts_buttocks
  body_parts_foot_sole
  body_parts_foot_top_or_side
  body_parts_genitalia_or_groin
  body_parts_head_or_neck
  body_parts_leg
  body_parts_other
  body_parts_palm
  body_parts_torso_back
  body_parts_torso_front
  case_id
  combined_race
  condition_duration
  condition_symptoms_bleeding
  condition_symptoms_bothersome_appearance
  condition_symptoms_burning
  condition_symptoms_darkening
  condition_symptoms_increasing_size
  condition_symptoms_itching
  condition_symptoms_no_relevant_experience
  condition_symptoms_pain
  dermatologist_fitzpatrick_skin_type_label_1
  dermatologist_fitzpatrick_skin_type_label_2
  dermatologist_fitzpatrick_skin_type_label_3
  dermatologist_gradable_for_fitzpatrick_skin_type_1
  dermatologist_gradable_for_fitzpatrick_skin_type_2
  dermatologist_gradable_for_fitzpatrick_skin_type_3
  dermatologist_gradable_for_skin_condition_1
  dermatologist_gradable_for_s

## 4. Precompute aligned face tensors  (~20-60 min)

Streams from HuggingFace, filters face/neck/scalp rows, runs:
`decode -> face-detect -> align -> CLAHE+WB -> save .npy`

Disk usage: only the .npy files (~200 MB for 1k faces, ~600 MB for 3k).
No parquet shards saved locally.

| `--source` | What streams |
|---|---|
| `scin_hf` | SCIN only |
| `fitzpatrick17k` | Fitzpatrick17k only |
| `all` | Both combined (recommended) |

Fully resumable - re-run to continue.

In [18]:
SOURCE = 'all'   # scin_hf | fitzpatrick17k | all

!python -m skin_training.data.precompute \
  --source {SOURCE} \
  --output /content/work \
  --aligned-size 256 \
  --verbose 2>&1 | tail -60

2026-04-27 09:06:44,779 INFO precompute loading source: scin_hf
2026-04-27 09:06:44,779 ERROR precompute source scin_hf failed: cannot import name 'FACE_BODY_PARTS' from 'skin_training.data.labels' (/content/skincare/backend_v2/ml_training/src/skin_training/data/labels.py)
Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/skincare/backend_v2/ml_training/src/skin_training/data/precompute.py", line 261, in <module>
    main()
  File "/content/skincare/backend_v2/ml_training/src/skin_training/data/precompute.py", line 248, in main
    precompute(
  File "/content/skincare/backend_v2/ml_training/src/skin_training/data/precompute.py", line 100, in precompute
    samples = load_samples(source, scin_root, hf_cache_dir, hf_token, face_only)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/skincare/backend_v2/ml_training/src/skin_training/da

In [19]:
import json, re, pandas as pd
idx = json.loads(open('/content/work/index.json').read())
print(json.dumps(idx, indent=2))
n, pct = idx['n_samples'], idx.get('face_detect_skip_pct','?')
print(f"\nAligned face tensors: {n}  |  face-detector skip rate: {pct}%")
df = pd.read_csv('/content/work/labels.csv')
cond_cols = [c for c in df.columns if re.match(r'^c\d+_', c)]
print("\nCondition coverage:")
for col in cond_cols:
    n_pos = int(df[col].sum())
    pct2  = 100 * n_pos / max(1, len(df))
    flag  = "[!] " if n_pos < 30 else "    "
    print(f"  {flag}{col:<28} {n_pos:>6} positives  ({pct2:.1f}%)")
if n < 300:
    print("\n[!] Very few samples - try --all-body-parts flag")

FileNotFoundError: [Errno 2] No such file or directory: '/content/work/index.json'

## 5. Configure training

In [ ]:
import yaml
from pathlib import Path
cfg = yaml.safe_load(Path('ml_training/configs/base.yaml').read_text())
cfg['data']['aligned_dir']          = '/content/work'
cfg['train']['checkpoint_dir']      = '/content/work/runs/exp1'
cfg['train']['batch_size']          = 32
cfg['train']['epochs']              = 25
cfg['train']['mixed_precision']     = True
cfg['train']['early_stop_patience'] = 6
Path('/content/config.yaml').write_text(yaml.safe_dump(cfg))
print(yaml.safe_dump(cfg))

## 6. Train  (~40-80 min on T4)
Resumable - interrupt and re-run to continue from last.pt.

In [ ]:
!python -m skin_training.train.loop --config /content/config.yaml --verbose

## 7. TensorBoard (run in a second tab while training)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/work/runs/exp1/tb

## 8. Export ONNX + roundtrip check

In [ ]:
!python -m skin_training.export.to_onnx \
  --checkpoint /content/work/runs/exp1/best.pt \
  --config /content/config.yaml \
  --output /content/work/skin_classifier_mobilenet.onnx \
  --verbose

## 9. Save to Drive + download
Drive is only used here (artifact output), never for input data.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/skincare/artifacts
!cp /content/work/skin_classifier_mobilenet.onnx /content/drive/MyDrive/skincare/artifacts/
!cp /content/work/runs/exp1/best.pt            /content/drive/MyDrive/skincare/artifacts/
!cp /content/work/index.json                   /content/drive/MyDrive/skincare/artifacts/
print('saved to Drive')
files.download('/content/work/skin_classifier_mobilenet.onnx')

## 10. Local install  (Windows machine)

```powershell
Copy-Item $HOME\Downloads\skin_classifier_mobilenet.onnx `
  C:\Users\Sozuri\skincare\backend_v2\ml_service\models\

cd C:\Users\Sozuri\skincare\backend_v2
docker compose build ml-service
docker compose up -d ml-service

curl http://localhost:8013/healthz   # expect models_loaded: true
```

After that POST /api/v1/analyze returns `"inference_mode": "onnx_mobilenet"`.

Report back: val_F1_macro + Fitzpatrick stratified block + index.json
Then I start Phase 6: MobileNet distillation + INT8 quant.